# ChurnIQ

## Phase 5: Feature Engineering

Objective:
Transform cleaned customer data into meaningful business features that improve churn prediction and business decision-making.

Current Dataset:
df_clean

Status:
- Missing values handled
- Data types reviewed
- Leakage review completed
- Ready for feature engineering

Author:
Akshit Sharma

In [1]:
# ==========================================
# Phase 5 - Feature Engineering
# ==========================================

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
import os

print(os.getcwd())

C:\Users\Akshit\churniq\notebooks


In [3]:
import os

print("Project Root:")
print(os.listdir(".."))

Project Root:
['.env', '.git', '.gitignore', '.ipynb_checkpoints', '.vscode', 'anaconda_projects', 'app', 'backup_ai_scaffold', 'config', 'data', 'database', 'models', 'notebooks', 'README.md', 'reports', 'requirements.txt']


In [4]:
print("\nData Folder:")
print(os.listdir("../data"))


Data Folder:
['processed', 'raw']


In [5]:
import os

print(os.listdir("../data/processed"))

['.gitkeep', 'train_clean.csv']


In [6]:
df_clean = pd.read_csv("../data/processed/train_clean.csv")

print(df_clean.shape)

(51047, 61)


In [7]:
print("Shape:", df_clean.shape)

print("\nTotal Missing Values:")
print(df_clean.isnull().sum().sum())

print("\nTarget Distribution:")
print(df_clean["Churn"].value_counts())

Shape: (51047, 61)

Total Missing Values:
0

Target Distribution:
Churn
No     36336
Yes    14711
Name: count, dtype: int64


In [8]:
# ==========================================
# ServiceArea Inspection
# ==========================================

print("Unique Service Areas:")
print(df_clean["ServiceArea"].nunique())

print("\nTop 20 Service Areas:")
print(df_clean["ServiceArea"].value_counts().head(20))

# ==========================================
# CreditRating Inspection
# ==========================================

print("\nCreditRating Distribution:")
print(df_clean["CreditRating"].value_counts(dropna=False))

Unique Service Areas:
748

Top 20 Service Areas:
ServiceArea
NYCBRO917    1684
HOUHOU281    1510
DALDAL214    1498
NYCMAN917    1182
APCFCH703     783
DALFTW817     782
SANSAN210     724
APCSIL301     670
SANAUS512     612
SFROAK510     605
SFRSFR415     565
NYCQUE917     533
ATLANE678     524
PHXPHX602     512
SFRSCL408     508
ATLATL678     507
STLSTL314     497
OHICOL614     486
MINMIN612     481
NEVLVS702     479
Name: count, dtype: int64

CreditRating Distribution:
CreditRating
2-High       18993
1-Highest     8522
3-Good        8410
5-Low         6499
4-Medium      5357
7-Lowest      2114
6-VeryLow     1152
Name: count, dtype: int64


## Step 5.2 - CreditRating Business Signal Validation

Objective:
Check whether customer credit quality is associated with churn behavior before deciding encoding strategy.

In [9]:
credit_churn = pd.crosstab(
    df_clean["CreditRating"],
    df_clean["Churn"],
    normalize="index"
) * 100

credit_churn.round(2)

Churn,No,Yes
CreditRating,,
1-Highest,69.16,30.84
2-High,69.93,30.07
3-Good,68.99,31.01
4-Medium,73.88,26.12
5-Low,77.90,22.10
6-VeryLow,72.57,27.43
7-Lowest,71.05,28.95


## Step 5.3 - ServiceArea Structure Investigation

Objective:
Understand the internal structure of ServiceArea before deciding an encoding strategy.

In [10]:
print("Sample ServiceArea Values:")
display(df_clean["ServiceArea"].head(20))

print("\nLength Distribution:")
display(df_clean["ServiceArea"].astype(str).str.len().value_counts())

Sample ServiceArea Values:


0     SEAPOR503
1     PITHOM412
2     MILMIL414
3     PITHOM412
4     OKCTUL918
5     OKCTUL918
6     OKCTUL918
7     OKCOKC405
8     SANMCA210
9     PITHOM412
10    SANMCA210
11    SLCSLC801
12    OKCOKC405
13    SLCSLC801
14    MILMIL414
15    LOULOU502
16    SLCSLC801
17    SANMCA210
18    KCYKCK913
19    SANMCA210
Name: ServiceArea, dtype: object


Length Distribution:


ServiceArea
9    51023
7       24
Name: count, dtype: int64

## ServiceArea Analysis

Investigating the structure of ServiceArea before feature engineering.

In [11]:
service_sample = pd.DataFrame()

service_sample["ServiceArea"] = df_clean["ServiceArea"].head(20)
service_sample["Part1"] = service_sample["ServiceArea"].str[:3]
service_sample["Part2"] = service_sample["ServiceArea"].str[3:6]
service_sample["AreaCode"] = service_sample["ServiceArea"].str[6:]

service_sample

,ServiceArea,Part1,Part2,AreaCode
0,SEAPOR503,SEA,POR,503
1,PITHOM412,PIT,HOM,412
2,MILMIL414,MIL,MIL,414
3,PITHOM412,PIT,HOM,412
4,OKCTUL918,OKC,TUL,918
5,OKCTUL918,OKC,TUL,918
6,OKCTUL918,OKC,TUL,918
7,OKCOKC405,OKC,OKC,405
8,SANMCA210,SAN,MCA,210
9,PITHOM412,PIT,HOM,412


## Revenue-Based Features

In [12]:
df_clean["RevenuePerMinute"] = (
    df_clean["MonthlyRevenue"] /
    (df_clean["MonthlyMinutes"] + 1)
)

In [13]:
df_clean["RevenuePerMinute"].describe()

count    51047.000000
mean         0.648927
std          3.705537
min         -6.170000
25%          0.088113
50%          0.136374
75%          0.246320
max        163.230000
Name: RevenuePerMinute, dtype: float64

## RevenuePerMinute Validation

In [14]:
df_clean[df_clean["RevenuePerMinute"] < 0][
    ["MonthlyRevenue", "MonthlyMinutes", "RevenuePerMinute"]
].head(10)

,MonthlyRevenue,MonthlyMinutes,RevenuePerMinute
26596,-2.52,211.0,-0.011887
33352,-5.86,0.0,-5.860000
48038,-6.17,0.0,-6.170000


In [15]:
print("Negative RevenuePerMinute Count:")
print((df_clean["RevenuePerMinute"] < 0).sum())

Negative RevenuePerMinute Count:
3


## RevenuePerMinute Outlier Check

In [16]:
df_clean.nlargest(
    10,
    "RevenuePerMinute"
)[
    ["MonthlyRevenue",
     "MonthlyMinutes",
     "RevenuePerMinute"]
]

,MonthlyRevenue,MonthlyMinutes,RevenuePerMinute
24285,163.23,0.0,163.23
44077,114.42,0.0,114.42
44452,106.24,0.0,106.24
5888,102.12,0.0,102.12
17288,98.24,0.0,98.24
2319,94.99,0.0,94.99
43059,94.97,0.0,94.97
23508,88.25,0.0,88.25
50733,87.49,0.0,87.49
20111,85.20,0.0,85.20


## MonthlyMinutes Validation

In [17]:
zero_minutes = (df_clean["MonthlyMinutes"] == 0).sum()

print("Customers with MonthlyMinutes = 0:")
print(zero_minutes)

Customers with MonthlyMinutes = 0:
723


## Revenue Analysis

In [18]:
df_clean["MonthlyRevenue"].describe()

count    51047.000000
mean        58.802788
std         44.442964
min         -6.170000
25%         33.660000
50%         48.460000
75%         70.960000
max       1223.380000
Name: MonthlyRevenue, dtype: float64

In [19]:
df_clean["MonthlyRevenue"].quantile(
    [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

0.25     33.660
0.50     48.460
0.75     70.960
0.90    103.654
0.95    135.045
0.99    225.668
Name: MonthlyRevenue, dtype: float64

## Revenue vs Churn Analysis

In [20]:
revenue_bins = pd.qcut(
    df_clean["MonthlyRevenue"],
    q=4,
    duplicates="drop"
)

revenue_churn = pd.crosstab(
    revenue_bins,
    df_clean["Churn"],
    normalize="index"
) * 100

revenue_churn.round(2)

Churn,No,Yes
MonthlyRevenue,,
"(-6.171, 33.66]",68.58,31.42
"(33.66, 48.46]",72.70,27.30
"(48.46, 70.96]",71.34,28.66
"(70.96, 1223.38]",72.09,27.91


## CurrentEquipmentDays Analysis

In [21]:
df_clean["CurrentEquipmentDays"].describe()

count    51047.000000
mean       380.544831
std        253.799599
min         -5.000000
25%        205.000000
50%        329.000000
75%        515.000000
max       1812.000000
Name: CurrentEquipmentDays, dtype: float64

In [22]:
df_clean["CurrentEquipmentDays"].quantile(
    [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

0.25     205.0
0.50     329.0
0.75     515.0
0.90     732.0
0.95     866.0
0.99    1147.0
Name: CurrentEquipmentDays, dtype: float64

## CurrentEquipmentDays Validation

In [23]:
print("Negative Values Count:")
print((df_clean["CurrentEquipmentDays"] < 0).sum())

print("\nNegative Value Records:")
display(
    df_clean.loc[
        df_clean["CurrentEquipmentDays"] < 0,
        ["CurrentEquipmentDays", "Churn"]
    ].head(10)
)

Negative Values Count:
76

Negative Value Records:


,CurrentEquipmentDays,Churn
107,-3.0,Yes
424,-1.0,No
2107,-3.0,No
2145,-1.0,No
3883,-3.0,No
3923,-2.0,No
4075,-5.0,No
4318,-2.0,No
4319,-1.0,No
4732,-3.0,No


## CurrentEquipmentDays Correction

In [24]:
# Replace invalid negative values

df_clean["CurrentEquipmentDays"] = df_clean["CurrentEquipmentDays"].replace(
    [-1, -2, -3, -4, -5],
    np.nan
)

# Median imputation

df_clean["CurrentEquipmentDays"] = df_clean["CurrentEquipmentDays"].fillna(
    df_clean["CurrentEquipmentDays"].median()
)

In [25]:
print("Negative Values Remaining:")
print((df_clean["CurrentEquipmentDays"] < 0).sum())

Negative Values Remaining:
0


## CurrentEquipmentDays vs Churn

In [26]:
equipment_bins = pd.qcut(
    df_clean["CurrentEquipmentDays"],
    q=4,
    duplicates="drop"
)

equipment_churn = pd.crosstab(
    equipment_bins,
    df_clean["Churn"],
    normalize="index"
) * 100

equipment_churn.round(2)

Churn,No,Yes
CurrentEquipmentDays,,
"(-0.001, 206.0]",78.03,21.97
"(206.0, 330.0]",75.11,24.89
"(330.0, 515.0]",65.21,34.79
"(515.0, 1812.0]",66.25,33.75


## Handset Behavior Analysis

In [27]:
df_clean[["Handsets", "HandsetModels"]].describe()

,Handsets,HandsetModels
count,51047.000000,51047.000000
mean,1.805630,1.558740
std,1.331165,0.905927
min,1.000000,1.000000
25%,1.000000,1.000000
50%,1.000000,1.000000
75%,2.000000,2.000000
max,24.000000,15.000000


## HandsetModels vs Churn

In [29]:
model_bins = pd.qcut(
    df_clean["HandsetModels"],
    q=4,
    duplicates="drop"
)

model_churn = pd.crosstab(
    model_bins,
    df_clean["Churn"],
    normalize="index"
) * 100

model_churn.round(2)

Churn,No,Yes
HandsetModels,,
"(0.999, 2.0]",70.63,29.37
"(2.0, 15.0]",75.22,24.78


## Feature Inventory Planning

In [30]:
df_clean.columns.tolist()

['CustomerID',
 'Churn',
 'MonthlyRevenue',
 'MonthlyMinutes',
 'TotalRecurringCharge',
 'DirectorAssistedCalls',
 'OverageMinutes',
 'RoamingCalls',
 'PercChangeMinutes',
 'PercChangeRevenues',
 'DroppedCalls',
 'BlockedCalls',
 'UnansweredCalls',
 'CustomerCareCalls',
 'ThreewayCalls',
 'ReceivedCalls',
 'OutboundCalls',
 'InboundCalls',
 'PeakCallsInOut',
 'OffPeakCallsInOut',
 'DroppedBlockedCalls',
 'CallForwardingCalls',
 'CallWaitingCalls',
 'MonthsInService',
 'UniqueSubs',
 'ActiveSubs',
 'ServiceArea',
 'Handsets',
 'HandsetModels',
 'CurrentEquipmentDays',
 'AgeHH1',
 'AgeHH2',
 'ChildrenInHH',
 'HandsetRefurbished',
 'HandsetWebCapable',
 'TruckOwner',
 'RVOwner',
 'Homeownership',
 'BuysViaMailOrder',
 'RespondsToMailOffers',
 'OptOutMailings',
 'NonUSTravel',
 'OwnsComputer',
 'HasCreditCard',
 'RetentionCalls',
 'RetentionOffersAccepted',
 'NewCellphoneUser',
 'NotNewCellphoneUser',
 'ReferralsMadeBySubscriber',
 'IncomeGroup',
 'OwnsMotorcycle',
 'AdjustmentsToCreditRat

In [31]:
df_clean.drop(columns=["RevenuePerMinute"], inplace=True)

In [32]:
"RevenuePerMinute" in df_clean.columns

False

## Feature Inventory

In [33]:
categorical_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = df_clean.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical Features:", len(categorical_cols))
print(categorical_cols)

print("\nNumerical Features:", len(numerical_cols))
print(numerical_cols)

Categorical Features: 22
['Churn', 'ServiceArea', 'ChildrenInHH', 'HandsetRefurbished', 'HandsetWebCapable', 'TruckOwner', 'RVOwner', 'Homeownership', 'BuysViaMailOrder', 'RespondsToMailOffers', 'OptOutMailings', 'NonUSTravel', 'OwnsComputer', 'HasCreditCard', 'NewCellphoneUser', 'NotNewCellphoneUser', 'OwnsMotorcycle', 'MadeCallToRetentionTeam', 'CreditRating', 'PrizmCode', 'Occupation', 'MaritalStatus']

Numerical Features: 39
['CustomerID', 'MonthlyRevenue', 'MonthlyMinutes', 'TotalRecurringCharge', 'DirectorAssistedCalls', 'OverageMinutes', 'RoamingCalls', 'PercChangeMinutes', 'PercChangeRevenues', 'DroppedCalls', 'BlockedCalls', 'UnansweredCalls', 'CustomerCareCalls', 'ThreewayCalls', 'ReceivedCalls', 'OutboundCalls', 'InboundCalls', 'PeakCallsInOut', 'OffPeakCallsInOut', 'DroppedBlockedCalls', 'CallForwardingCalls', 'CallWaitingCalls', 'MonthsInService', 'UniqueSubs', 'ActiveSubs', 'Handsets', 'HandsetModels', 'CurrentEquipmentDays', 'AgeHH1', 'AgeHH2', 'RetentionCalls', 'Retenti

## MonthsInService Analysis

In [34]:
df_clean["MonthsInService"].describe()

count    51047.000000
mean        18.756264
std          9.800138
min          6.000000
25%         11.000000
50%         16.000000
75%         24.000000
max         61.000000
Name: MonthsInService, dtype: float64

In [35]:
months_churn = pd.crosstab(
    pd.qcut(df_clean["MonthsInService"], q=4, duplicates="drop"),
    df_clean["Churn"],
    normalize="index"
) * 100

months_churn.round(2)

Churn,No,Yes
MonthsInService,,
"(5.999, 11.0]",75.97,24.03
"(11.0, 16.0]",65.89,34.11
"(16.0, 24.0]",70.20,29.80
"(24.0, 61.0]",71.54,28.46


## Customer Lifecycle Stage

In [36]:
df_clean["CustomerLifecycleStage"] = pd.cut(
    df_clean["MonthsInService"],
    bins=[0, 12, 24, float("inf")],
    labels=[
        "New",
        "Growing",
        "Loyal"
    ]
)

In [37]:
df_clean["CustomerLifecycleStage"].value_counts()

CustomerLifecycleStage
Growing    21422
New        16975
Loyal      12650
Name: count, dtype: int64

In [38]:
pd.crosstab(
    df_clean["CustomerLifecycleStage"],
    df_clean["Churn"],
    normalize="index"
) * 100

Churn,No,Yes
CustomerLifecycleStage,,
New,73.372607,26.627393
Growing,69.232565,30.767435
Loyal,71.541502,28.458498


## Engineered Features Log

In [39]:
engineered_features = [
    "CustomerLifecycleStage"
]

engineered_features

['CustomerLifecycleStage']

## Customer Value Proxy

In [40]:
df_clean["CustomerValueProxy"] = (
    df_clean["MonthlyRevenue"] *
    df_clean["MonthsInService"]
)

In [41]:
df_clean["CustomerValueProxy"].describe()

count    51047.000000
mean      1101.916733
std       1148.641665
min        -87.900000
25%        470.860000
50%        784.700000
75%       1325.300000
max      38066.400000
Name: CustomerValueProxy, dtype: float64

In [42]:
df_clean["CustomerValueProxy"] = (
    df_clean["MonthlyRevenue"] *
    df_clean["MonthsInService"]
)

df_clean["CustomerValueProxy"].describe()

count    51047.000000
mean      1101.916733
std       1148.641665
min        -87.900000
25%        470.860000
50%        784.700000
75%       1325.300000
max      38066.400000
Name: CustomerValueProxy, dtype: float64

## Customer Value Proxy Validation

In [43]:
value_bins = pd.qcut(
    df_clean["CustomerValueProxy"],
    q=4,
    duplicates="drop"
)

value_churn = pd.crosstab(
    value_bins,
    df_clean["Churn"],
    normalize="index"
) * 100

value_churn.round(2)

Churn,No,Yes
CustomerValueProxy,,
"(-87.90100000000001, 470.86]",73.60,26.40
"(470.86, 784.7]",70.32,29.68
"(784.7, 1325.3]",69.22,30.78
"(1325.3, 38066.4]",71.58,28.42


## CreditRating Encoding

In [44]:
credit_rating_map = {
    "1-Highest": 7,
    "2-High": 6,
    "3-Good": 5,
    "4-Medium": 4,
    "5-Low": 3,
    "6-VeryLow": 2,
    "7-Lowest": 1
}

df_clean["CreditRating_Encoded"] = (
    df_clean["CreditRating"]
    .map(credit_rating_map)
)

df_clean[
    ["CreditRating", "CreditRating_Encoded"]
].head()

,CreditRating,CreditRating_Encoded
0,1-Highest,7
1,4-Medium,4
2,3-Good,5
3,4-Medium,4
4,1-Highest,7


## ServiceArea Component Analysis

In [45]:
df_clean["RegionCode"] = df_clean["ServiceArea"].str[:3]
df_clean["MarketCode"] = df_clean["ServiceArea"].str[3:6]
df_clean["AreaCode"] = df_clean["ServiceArea"].str[6:]

print("RegionCode:", df_clean["RegionCode"].nunique())
print("MarketCode:", df_clean["MarketCode"].nunique())
print("AreaCode:", df_clean["AreaCode"].nunique())

RegionCode: 58
MarketCode: 534
AreaCode: 197


In [46]:
df_clean[["RegionCode", "MarketCode", "AreaCode"]].head()

,RegionCode,MarketCode,AreaCode
0,SEA,POR,503
1,PIT,HOM,412
2,MIL,MIL,414
3,PIT,HOM,412
4,OKC,TUL,918


## Cardinality Check

In [47]:
for col in ["PrizmCode", "Occupation", "MaritalStatus"]:
    print(f"\n{col}")
    print(df_clean[col].value_counts(dropna=False))


PrizmCode
PrizmCode
Other       24655
Suburban    16378
Town         7589
Rural        2425
Name: count, dtype: int64

Occupation
Occupation
Other           37637
Professional     8755
Crafts           1519
Clerical          986
Self              879
Retired           733
Student           381
Homemaker         157
Name: count, dtype: int64

MaritalStatus
MaritalStatus
Unknown    19700
Yes        18651
No         12696
Name: count, dtype: int64


## Leakage Feature Review

In [48]:
leakage_features = [
    "RetentionCalls",
    "RetentionOffersAccepted",
    "MadeCallToRetentionTeam"
]

for col in leakage_features:
    print(f"\n{col}")
    print(pd.crosstab(
        df_clean[col],
        df_clean["Churn"],
        normalize="index"
    ) * 100)


RetentionCalls
Churn                  No         Yes
RetentionCalls                       
0               71.755710   28.244290
1               55.003108   44.996892
2               55.833333   44.166667
3               50.000000   50.000000
4                0.000000  100.000000

RetentionOffersAccepted
Churn                           No        Yes
RetentionOffersAccepted                      
0                        71.396962  28.603038
1                        59.020311  40.979689
2                        55.555556  44.444444
3                        62.500000  37.500000

MadeCallToRetentionTeam
Churn                          No       Yes
MadeCallToRetentionTeam                    
No                       71.75571  28.24429
Yes                      54.95702  45.04298


## Feature Freeze

In [50]:
df_clean.to_csv(
    "../data/processed/feature_engineered.csv",
    index=False
)

In [51]:
print("Shape:", df_clean.shape)

print("\nEngineered Features:")
engineered_features = [
    "CustomerLifecycleStage",
    "CreditRating_Encoded",
    "RegionCode",
    "AreaCode"
]

for feature in engineered_features:
    print("-", feature)

Shape: (51047, 67)

Engineered Features:
- CustomerLifecycleStage
- CreditRating_Encoded
- RegionCode
- AreaCode


## Save Feature Engineered Dataset

In [52]:
df_clean.to_csv(
    "../data/processed/feature_engineered.csv",
    index=False
)

print("feature_engineered.csv saved successfully")

feature_engineered.csv saved successfully


In [53]:
import os

print(os.listdir("../data/processed"))

['.gitkeep', 'feature_engineered.csv', 'train_clean.csv']


In [55]:
df_clean.to_csv(
    "../data/processed/feature_engineered.csv",
    index=False
)